# Electride screening — feature dataset and symbolic-model evaluator

**Three independent steps in one notebook:** (1) RAG feature catalog (already provided as `electride_feature_rag_clean.jsonl`), (2) CIF → labeled CSV using **five individual tools**, (3) Pydantic symbolic classifier → safe equation fitting, metrics, correlations, permutation importance, textual feedback.

Place `electride_features.py`, `labels.json`, `structures/`, and `electride_feature_rag_clean.jsonl` next to this notebook. The evaluator **does not call an LLM**; the planning swarm can fill the Pydantic schema using its existing LLM connection. Input labels are supplied; they are never inferred from CIFs.

Install if necessary: `%pip install pymatgen ase scipy numpy pandas pydantic scikit-learn sympy tqdm`.

## 1. RAG feature capabilities

Index each JSONL line individually using your existing PDF/RAG retriever or embedding index. Return the top-matching records to the planning agent. Each record contains only `function`, `feature`, and `description`. The LLM chooses feature pairs; the evaluator retrieves `function__feature` from the CSV.

In [ ]:
import json
from pathlib import Path

RAG_FILE = Path("../electride_feature_rag_clean.jsonl")
FEATURE_CAPABILITIES = [json.loads(line) for line in RAG_FILE.read_text().splitlines() if line.strip()]
print(f"RAG-ready feature records: {len(FEATURE_CAPABILITIES)}")
print(json.dumps(FEATURE_CAPABILITIES[0], indent=2))

## 2. Extract the labeled feature CSV

One row per CIF. Runs **only** `structure_summary`, `electron_count_hypotheses`, `bonding_environment`, `cavity_geometry`, and `void_connectivity`. Each resulting column is `function__feature` (nested dictionary keys add `__`). Lists remain JSON strings. A failed function is recorded with `function__status` and `function__error`; no labels are fabricated.

In [ ]:
import importlib.util
import math
import pandas as pd
from tqdm.auto import tqdm

STRUCTURES_DIR = Path("structures")
LABELS_JSON = Path("labels.json")
FEATURE_MODULE = Path("electride_features.py")
DATASET_CSV = Path("electride_features_dataset.csv")

TOOL_NAMES = [
    "structure_summary",
    "electron_count_hypotheses",
    "bonding_environment",
    "cavity_geometry",
    "void_connectivity",
]



def flatten_output(obj, prefix):
    if isinstance(obj, dict):
        out = {}
        for key, value in obj.items():
            out.update(flatten_output(value, f"{prefix}__{key}"))
        return out
    if isinstance(obj, (list, tuple)):
        return {prefix: json.dumps(obj, default=str, allow_nan=False)}
    return {prefix: obj}


def create_feature_csv(structures_dir=STRUCTURES_DIR,
                       labels_file=LABELS_JSON,
                       output_csv=DATASET_CSV):
    """Extract all individual scientific tools for labeled CIFs; preserve errors."""
    # Import only when extracting CIFs; CSV-only evaluation needs no CIF dependencies.
    spec = importlib.util.spec_from_file_location("electride_features", FEATURE_MODULE)
    feature_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(feature_module)
    assert all(name in feature_module.TOOL_FUNCTIONS for name in TOOL_NAMES)
    labels = json.loads(Path(labels_file).read_text())
    label_by_name = {}
    for item in labels:
        filename = Path(item["structure"]).name
        raw = str(item["label"]).lower().strip()
        if raw not in {"yes", "no"}:
            raise ValueError(f"Unrecognized label {raw!r} for {filename}")
        if filename in label_by_name:
            raise ValueError(f"Duplicate structure label: {filename}")
        label_by_name[filename] = int(raw == "yes")

    paths = sorted(Path(structures_dir).rglob("*.cif"))
    if not paths:
        raise FileNotFoundError(f"No CIF files found in {structures_dir}")
    rows = []
    for path in tqdm(paths, desc="CIF features"):
        if path.name not in label_by_name:
            print(f"Unlabeled CIF, skipped: {path.name}")
            continue
        y = label_by_name[path.name]
        row = {"structure": path.name, "cif_path": str(path),
               "label": y, "class": "electride" if y else "non-electride"}
        used=[]
        for name in TOOL_NAMES:
            print(f"[{path.name}] {name}")
            try:
                result = feature_module.dispatch_tool(name, {"cif_path": str(path)})
                row.update(flatten_output(result, name))
                row[f"{name}__status"] = "success"
                used.append(name)
            except Exception as exc:
                row[f"{name}__status"] = "failed"
                row[f"{name}__error"] = f"{type(exc).__name__}: {exc}"
                print("   FAILED:", row[f"{name}__error"])
        row["tools_used"] = json.dumps(used)
        rows.append(row)

    dataset = pd.DataFrame(rows)
    if dataset.empty:
        raise ValueError("No labeled CIF structures could be processed")
    dataset.to_csv(output_csv, index=False)
    print(f"Saved {len(dataset)} structures x {len(dataset.columns)} columns to {output_csv}")
    print(dataset["label"].value_counts().to_dict())
    missing = sorted(set(label_by_name) - {p.name for p in paths})
    if missing:
        print(f"Warning: {len(missing)} labeled structures have no matching CIF file")
    return dataset

# Run when CIF files are available:
df = create_feature_csv()